## Layers & the union filesystem

Every filesystem-changing instruction in a Dockerfile (`FROM`, `RUN`, `COPY`, `ADD`) usually produces a new **layer** — a tarball of "what changed relative to the layer below me." Layers stack bottom-up, and a **union filesystem** driver in the daemon presents them to the container as one root filesystem.

```
   +-----------------------------+
   |  writable container layer   |  <- a running container's changes go here
   +-----------------------------+
   |  L4: COPY app.py /app/      |  read-only
   |  L3: pip install flask      |  read-only
   |  L2: apt install python3    |  read-only
   |  L1: debian:bookworm rootfs |  read-only
   +-----------------------------+
```

The default driver on Linux is **`overlay2`**, which uses the kernel's overlayfs to merge the directories with **copy-on-write** semantics:

- **Reads** fall through the stack — a file is served from the topmost layer that has it.
- **Writes** trigger a *copy-up* — the file is copied from its read-only layer into the writable layer, then modified there; the original is untouched.
- **Deletes** are recorded as *whiteout files* in the writable layer that mask entries below.

Two consequences that shape everything you do:

1. **Layers are cached.** Re-run `docker build` and, if instructions 1–N haven't changed, the daemon reuses their existing layers instead of rebuilding — the foundation of fast builds (next sections lean on this hard).
2. **Nothing is ever truly deleted from a lower layer.** `RUN rm bigfile` in a later layer just adds a whiteout — the bytes still sit in the layer that added them, still shipped. That's why you clean up *within the same `RUN`* that created the mess, not in a later one.

**The writable layer is ephemeral:** delete the container and it's gone. Persisting data is what volumes are for (module 04).